# 04 - Synthetic DFIT Data Generation

# DFIT Pressure Diagnostics
Developed as part of DFIT pressure-diagnostics research in the Harold Vance Department of Petroleum Engineering at Texas A&M University.

Primary reference: Barree, Miskimins & Gilbert (2015), SPE-169539-PA.

Real DFIT data are proprietary, which makes interpretation software hard to test and teach. The synthetic generator forward-models physically reasonable pressure-time records for each leakoff regime, with known ground truth, so the whole toolkit can be exercised without confidential data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import dfit

## How it works

1. A regime-specific positive shape s(G) is defined for the pre-closure pressure-drop derivative dD/dG.
2. The shape is scaled so the pressure drop integrates exactly to ISIP - closure_pressure at the specified closure_G.
3. An after-closure reservoir decline (linear or radial) is appended.
4. G is mapped back to physical time via the inverse G-function, an injection ramp is prepended, and gauge noise is added.

In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
for regime in dfit.REGIMES:
    d = dfit.generate_dfit(regime=regime, closure_G=6.0, seed=1)
    fall = np.isfinite(d.G)
    ax.plot(d.G[fall], d.pressure_psi[fall], lw=1, label=regime)
ax.axvline(6.0, color='k', ls=':', alpha=0.5)
ax.set_xlabel('G-function'); ax.set_ylabel('pressure (psi)')
ax.set_title('Synthetic falloffs, four leakoff regimes')
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

## Controlling the test

Every physical knob is exposed: pump time, ISIP, closure pressure and G, reservoir pressure, after-closure regime, noise, and the leakoff exponent alpha.

In [ ]:
d = dfit.generate_dfit(regime='pressure_dependent',
                       t_pump_min=4.0, isip_psi=9200,
                       closure_pressure_psi=7600, closure_G=8.0,
                       reservoir_pressure_psi=6100,
                       after_closure_regime='linear',
                       noise_psi=2.0, seed=15)
df = dfit.to_dataframe(d)
df.head()

In [ ]:
# round-trip: feed the synthetic test back through the interpreter
res = dfit.analyze_dfit(d.time_min, d.pressure_psi, d.rate_bpm)
print(res.summary())
print()
print('truth closure_G =', d.truth['closure_G'],
      '| net pressure =', d.truth['net_pressure_psi'])

### Use it for

- unit-testing every interpretation routine against known answers,
- teaching the diagnostic signatures, and
- benchmarking how noise and sampling degrade the closure pick.